# 觀察咖啡豆：好豆和壞豆的顏色到底差在哪？

上一個 notebook 我們把整盤豆子切成一顆一顆。
這次我們要當偵探，仔細「看」這些豆子的顏色，找出好豆和壞豆的差別。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arc13Tangent/Pu-Tai-Coffee-Machine-Learning/blob/main/From_Cropped_to_Features.ipynb)

## 步驟 0：下載課程的豆子照片

按下面這一格的 ▶（或 `Shift + Enter`），把切好的豆子照片從 GitHub 下載下來。
看到 `✅ 準備完成！` 就成功了。

In [ ]:
# 下載課程資料
![ -d Pu-Tai-Coffee-Machine-Learning ] && echo '已經載過了，跳過' || git clone --depth 1 https://github.com/Arc13Tangent/Pu-Tai-Coffee-Machine-Learning.git

# 套件
import cv2
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from plotly.subplots import make_subplots

# 豆子照片的位置
BASE = Path("Pu-Tai-Coffee-Machine-Learning/Images/Cropped")
GOOD_DIR = BASE / "Normal_Beans"          # 好豆
BAD_DIR  = BASE / "Defective_Beans"       # 壞豆
GOOD_MASK_DIR = BASE / "Normal_Beans_Mask"   # 好豆的「形狀遮罩」（白色=豆子，黑色=背景）
BAD_MASK_DIR  = BASE / "Defective_Beans_Mask"

# 把每個資料夾的檔名列出來、排好序
good_files = sorted(p.name for p in GOOD_DIR.glob("*.jpg"))
bad_files  = sorted(p.name for p in BAD_DIR.glob("*.jpg"))

print(f"✅ 準備完成！")
print(f"   好豆有 {len(good_files)} 顆")
print(f"   壞豆有 {len(bad_files)} 顆")

## 步驟 1：選一顆好豆、一顆壞豆，仔細看它們的顏色

改下面的 `GOOD_INDEX`（好豆第幾顆）和 `BAD_INDEX`（壞豆第幾顆），
執行後會並排顯示這兩顆豆子。

**把滑鼠移到豆子上面**，右上角會即時顯示那個點的位置和 RGB 顏色值。
試試看：移到豆子最深的地方、最亮的地方、還有背景（白色區域），
看看 RGB 三個數字怎麼變。

In [ ]:
# === 改這兩個數字，選你想看的豆子 ===
GOOD_INDEX = 0   # 好豆第幾顆（0 是第一顆）
BAD_INDEX  = 0   # 壞豆第幾顆
# =====================================

def load_rgb(folder, fname):
    """讀一張豆子圖，轉成正常的 RGB 顏色"""
    img = cv2.imread(str(folder / fname))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # OpenCV 是 BGR，要轉成 RGB

good_img = load_rgb(GOOD_DIR, good_files[GOOD_INDEX])
bad_img  = load_rgb(BAD_DIR,  bad_files[BAD_INDEX])

# 並排畫兩顆豆子
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=(f"好豆 #{GOOD_INDEX}", f"壞豆 #{BAD_INDEX}"))
fig.add_trace(go.Image(z=good_img), row=1, col=1)
fig.add_trace(go.Image(z=bad_img),  row=1, col=2)

fig.update_layout(height=450, title_text="把滑鼠移到豆子上，看看每個點的 RGB 值")
fig.show()

## 步驟 2：把很多顆豆子疊在一起，看顏色分佈的「形狀」

剛剛我們一個點一個點看，現在換個方式：
把好豆、壞豆**各隨機抓 N 顆**，算出它們每個顏色值（0~255）各有多少像素，
畫成曲線。紅、綠、藍三個顏色各一張圖。

- 橫軸：顏色的亮度（0 = 全黑，255 = 全亮）
- 縱軸：有多少比例的像素是這個亮度

**觀察重點**：好豆的曲線（藍線）和壞豆的曲線（紅線）形狀一樣嗎？
誰比較「高瘦集中」、誰比較「矮胖分散」？想想看為什麼。

改 `SEED` 可以換一批不同的豆子，改 `N` 可以調抓幾顆。

In [ ]:
# === 可以改這兩個數字 ===
SEED = 710   # 換一個數字 = 換一批隨機抓到的豆子
N    = 50   # 好豆、壞豆各抓幾顆
# ========================

def bean_pixels(img_dir, mask_dir, fname):
    """讀一顆豆子，只取『豆子本體』的像素（用遮罩去掉白色背景）"""
    img  = cv2.cvtColor(cv2.imread(str(img_dir / fname)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_dir / fname), cv2.IMREAD_GRAYSCALE)
    return img[mask > 127]

# 隨機抓 N 顆好豆、N 顆壞豆（步驟 3 會用同一批）
rng = np.random.default_rng(SEED)
good_pick = rng.choice(len(good_files), N, replace=False)
bad_pick  = rng.choice(len(bad_files),  N, replace=False)

bins    = np.arange(0, 257, 4)
centers = (bins[:-1] + bins[1:]) / 2
channel_names = ["紅 (R)", "綠 (G)", "藍 (B)"]

# 2 排 × 3 欄：上排好豆、下排壞豆
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f"好豆 - {c}" for c in channel_names] +
                   [f"壞豆 - {c}" for c in channel_names])

def draw_class(picks, files, img_dir, mask_dir, row, color):
    """把這一類的每一顆豆子，各畫一條曲線（同一排的三欄分別是 R/G/B）"""
    for i in picks:
        px = bean_pixels(img_dir, mask_dir, files[i])
        for ch in range(3):
            hist, _ = np.histogram(px[:, ch], bins=bins, density=True)
            fig.add_trace(
                go.Scatter(x=centers, y=hist, mode="lines",
                           line=dict(color=color, width=1),
                           opacity=0.6, showlegend=False),
                row=row, col=ch + 1)

draw_class(good_pick, good_files, GOOD_DIR, GOOD_MASK_DIR, row=1, color="royalblue")
draw_class(bad_pick,  bad_files,  BAD_DIR,  BAD_MASK_DIR,  row=2, color="crimson")

fig.update_layout(height=650,
                  title_text=f"每一顆豆子一條曲線（各 {N} 顆，SEED={SEED}）")
fig.show()

## 步驟 3：把每一顆豆子變成平面上的一個點

每一顆豆子，我們算它的兩個數字：
- **平均值**：這顆豆子的顏色「平均有多亮」
- **標準差**：這顆豆子的顏色「有多不均勻」（越大代表越花、越多斑點）

把「平均值」當橫軸、「標準差」當縱軸，每顆豆子就變成平面上的一個點。
看看好豆（藍點）和壞豆（紅點）會不會自己分成兩群。

下面的圖會把 **散布圖** 和 **豆子照片** 連動：
滑鼠移到某個點，對應的豆子會亮起來；移到某顆豆子，它的點也會亮起來。

In [ ]:
# 註：good_pick / bad_pick / N / SEED 沿用步驟 2 那批豆子
import base64, json

def bean_record(label, img_dir, mask_dir, fname):
    """算一顆豆子在 R/G/B 三個通道的 平均值、標準差，並做出可內嵌的縮圖"""
    img  = cv2.imread(str(img_dir / fname))
    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_dir / fname), cv2.IMREAD_GRAYSCALE)
    px   = rgb[mask > 127]                       # 只取豆子本體 (N,3)
    ok, buf = cv2.imencode(".jpg", img)          # imencode 吃 BGR，瀏覽器顯示正常
    return {
        "label": label,
        "name":  fname,
        "mean":  [float(px[:, c].mean()) for c in range(3)],   # [R, G, B]
        "std":   [float(px[:, c].std())  for c in range(3)],
        "img":   "data:image/jpeg;base64," + base64.b64encode(buf).decode(),
    }

beans = []
for i in good_pick:
    beans.append(bean_record("good", GOOD_DIR, GOOD_MASK_DIR, good_files[i]))
for i in bad_pick:
    beans.append(bean_record("bad",  BAD_DIR,  BAD_MASK_DIR,  bad_files[i]))

print(f"✅ 準備好 {len(beans)} 顆豆子的資料（R/G/B 三個通道都算好了）")

In [ ]:
# ===== 版本 Default：Plotly FigureWidget（語法是 Python，失靈請跑下一格）=====
from google.colab import output
output.enable_custom_widget_manager()
import plotly.graph_objects as go
from IPython.display import display, Image as IPyImage
import ipywidgets as widgets

colors = ["#4169e1" if b["label"]=="good" else "#dc143c" for b in beans]
names  = [("好豆" if b["label"]=="good" else "壞豆") for b in beans]
channel_names = ["紅 (R)", "綠 (G)", "藍 (B)"]

def make_hover(ch, out):
    def on_hover(trace, points, state):
        if not points.point_inds:
            return
        i = points.point_inds[0]
        out.clear_output(wait=True)
        with out:
            b = beans[i]
            print(f"{names[i]}")
            print(f"{channel_names[ch]}　平均={b['mean'][ch]:.1f}　標準差={b['std'][ch]:.1f}")
            raw = base64.b64decode(b["img"].split(",")[1])
            display(IPyImage(data=raw, width=120))
    return on_hover

rows = []
for ch in range(3):
    fw = go.FigureWidget()
    fw.add_scatter(
        x=[b["mean"][ch] for b in beans],
        y=[b["std"][ch]  for b in beans],
        mode="markers",
        marker=dict(size=10, color=colors, opacity=0.8),
        text=names, hoverinfo="text")
    fw.update_layout(width=460, height=300,
                     margin=dict(l=60, r=20, t=40, b=45),
                     xaxis_title="平均值（多亮）",
                     yaxis_title="標準差（多不均勻）",
                     title=f"{channel_names[ch]}")

    out = widgets.Output(layout=widgets.Layout(width="160px"))  # 這一列專屬的照片區
    fw.data[0].on_hover(make_hover(ch, out))
    rows.append(widgets.HBox([fw, out]))   # 圖 + 它右邊的照片區

display(widgets.VBox(rows))

In [ ]:
# ===== 版本 Fallback：純前端 HTML，R/G/B 三列，每列散布圖↔豆子雙向連動 =====
from IPython.display import HTML, display

payload = json.dumps(beans)
ranges = json.dumps({
    "mlo": [min(b["mean"][c] for b in beans) for c in range(3)],
    "mhi": [max(b["mean"][c] for b in beans) for c in range(3)],
    "slo": [min(b["std"][c]  for b in beans) for c in range(3)],
    "shi": [max(b["std"][c]  for b in beans) for c in range(3)],
})

html = f"""
<div id="app" style="font-family:sans-serif;"></div>
<script>
(function() {{
  const beans = {payload};
  const R = {ranges};
  const channelNames = ["紅 (R)", "綠 (G)", "藍 (B)"];
  const palette = {{good:"#4169e1", bad:"#dc143c"}};
  const labelsZh = {{good:"好豆", bad:"壞豆"}};
  const W=460,H=300,PADL=60,PADB=45,PADT=40,PADR=20;
  const SVGNS="http://www.w3.org/2000/svg";
  const app=document.getElementById("app");

  // 雙層框：內白外黑，黑底白底都看得見
  const RING_ON  = "3px solid #fff";          // outline（白）
  const SHADOW_ON= "0 0 0 5px #000";          // box-shadow（黑）

  channelNames.forEach((cname, ch) => {{
    const rowWrap=document.createElement("div");
    rowWrap.style.cssText="display:flex;align-items:flex-start;gap:16px;margin-bottom:18px;";

    const svg=document.createElementNS(SVGNS,"svg");
    svg.setAttribute("width",W);svg.setAttribute("height",H);
    svg.style.border="1px solid #ddd";
    rowWrap.appendChild(svg);

    // 右側：放大照片 + 資訊 + 橫向豆子陣列
    const side=document.createElement("div");
    side.style.cssText="display:flex;flex-direction:column;gap:8px;";

    const topRow=document.createElement("div");
    topRow.style.cssText="display:flex;align-items:center;gap:12px;height:130px;";
    const bigImg=document.createElement("img");
    bigImg.width=120;bigImg.height=120;
    bigImg.style.cssText="object-fit:cover;border-radius:6px;display:none;";
    const infoBox=document.createElement("div");
    infoBox.style.cssText="font-size:14px;color:#888;";
    infoBox.textContent="把滑鼠移到點或豆子上";
    topRow.appendChild(bigImg);topRow.appendChild(infoBox);
    side.appendChild(topRow);

    // 豆子陣列：橫向，每列 10 顆（20 顆 → 2 列），寬扁
    const strip=document.createElement("div");
    strip.style.cssText="display:grid;grid-template-columns:repeat(10,36px);gap:5px;";
    side.appendChild(strip);

    rowWrap.appendChild(side);
    app.appendChild(rowWrap);

    const mpad=(R.mhi[ch]-R.mlo[ch])*0.1+1, spad=(R.shi[ch]-R.slo[ch])*0.1+1;
    const mlo=R.mlo[ch]-mpad, mhi=R.mhi[ch]+mpad, slo=R.slo[ch]-spad, shi=R.shi[ch]+spad;
    const sx=m=>PADL+(m-mlo)/(mhi-mlo)*(W-PADL-PADR);
    const sy=s=>H-PADB-(s-slo)/(shi-slo)*(H-PADT-PADB);

    function L(x1,y1,x2,y2){{const l=document.createElementNS(SVGNS,"line");
      l.setAttribute("x1",x1);l.setAttribute("y1",y1);l.setAttribute("x2",x2);l.setAttribute("y2",y2);
      l.setAttribute("stroke","#999");svg.appendChild(l);}}
    L(PADL,PADT,PADL,H-PADB); L(PADL,H-PADB,W-PADR,H-PADB);
    function T(x,y,s,size,anchor,rot){{const t=document.createElementNS(SVGNS,"text");
      t.setAttribute("x",x);t.setAttribute("y",y);t.setAttribute("font-size",size||12);
      t.setAttribute("fill","#888");t.setAttribute("text-anchor",anchor||"middle");
      if(rot)t.setAttribute("transform",`rotate(-90 ${{x}} ${{y}})`);
      t.textContent=s;svg.appendChild(t);}}
    T(W/2,22,cname,15);
    T(W/2,H-8,"平均值（多亮）");
    T(16,H/2,"標準差（多不均勻）",12,"middle",true);

    const dots=[],thumbs=[];
    function highlight(i,on){{
      dots[i].setAttribute("r",on?9:5);
      dots[i].setAttribute("stroke",on?"#fff":"none");   // 內層白
      dots[i].setAttribute("stroke-width",on?3:0);
      // 外層黑：用 drop-shadow 在白圈外再描一圈黑
      dots[i].style.filter = on
        ? "drop-shadow(0 0 1px #000) drop-shadow(0 0 1px #000)"
        : "none";
      thumbs[i].style.outline   = on?RING_ON:"3px solid transparent";
      thumbs[i].style.boxShadow = on?SHADOW_ON:"none";
      if(on){{const b=beans[i];
        bigImg.src=b.img;bigImg.style.display="block";
        infoBox.style.color="#888";
        infoBox.textContent=`${{labelsZh[b.label]}}　平均=${{b.mean[ch].toFixed(1)}}　標準差=${{b.std[ch].toFixed(1)}}`;
      }}
    }}

    beans.forEach((b,i)=>{{
      const c=document.createElementNS(SVGNS,"circle");
      c.setAttribute("cx",sx(b.mean[ch]));c.setAttribute("cy",sy(b.std[ch]));
      c.setAttribute("r",5);c.setAttribute("fill",palette[b.label]);
      c.setAttribute("opacity",0.85);c.style.cursor="pointer";
      c.addEventListener("mouseenter",()=>highlight(i,true));
      c.addEventListener("mouseleave",()=>highlight(i,false));
      svg.appendChild(c);dots.push(c);

      const im=document.createElement("img");
      im.src=b.img;im.width=36;im.height=36;
      im.style.cssText="object-fit:cover;border-radius:3px;outline:3px solid transparent;cursor:pointer;";
      im.addEventListener("mouseenter",()=>highlight(i,true));
      im.addEventListener("mouseleave",()=>highlight(i,false));
      strip.appendChild(im);thumbs.push(im);
    }});
  }});
}})();
</script>
"""
display(HTML(html))

# 步驟 4：讓電腦自己找出那條分隔線（SVM）

剛剛你應該發現：在每個平面上，好豆和壞豆**大致**可以用一條直線分開，
只是中間有點重疊、不完美。

現在我們不要自己畫線，讓一個叫 **SVM（支持向量機）** 的方法，
自動找出「最好的那條線」。它的想法很單純：

> 找一條線，讓兩邊的點離這條線**越遠越好**——
> 就像在兩群人中間鋪一條最寬的路。

下面先準備多一點豆子來訓練（之前只用 10 顆太少了）。

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# === 可以改 ===
N_PER_CLASS = 500   # 好豆、壞豆各拿幾顆來訓練（最多就是資料夾裡的數量）
DATA_SEED   = 0
# ==============

def bean_features(img_dir, mask_dir, fname):
    """回傳這顆豆子的 6 個特徵：R/G/B 各自的 平均值 與 標準差"""
    img  = cv2.cvtColor(cv2.imread(str(img_dir / fname)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_dir / fname), cv2.IMREAD_GRAYSCALE)
    px   = img[mask > 127]
    means = [px[:, c].mean() for c in range(3)]
    stds  = [px[:, c].std()  for c in range(3)]
    return means + stds        # [R平均, G平均, B平均, R標準差, G標準差, B標準差]

rng = np.random.default_rng(DATA_SEED)
g_idx = rng.choice(len(good_files), min(N_PER_CLASS, len(good_files)), replace=False)
b_idx = rng.choice(len(bad_files),  min(N_PER_CLASS, len(bad_files)),  replace=False)

X, y = [], []
for i in g_idx:
    X.append(bean_features(GOOD_DIR, GOOD_MASK_DIR, good_files[i])); y.append(0)  # 0 = 好豆
for i in b_idx:
    X.append(bean_features(BAD_DIR,  BAD_MASK_DIR,  bad_files[i]));  y.append(1)  # 1 = 壞豆

X = np.array(X)
y = np.array(y)

# 切成訓練 / 測試，之後才能誠實地報「沒看過的豆子」準確率
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)

channel_names = ["紅 (R)", "綠 (G)", "藍 (B)"]
print(f"✅ 準備好 {len(X)} 顆豆子（好豆 {sum(y==0)}、壞豆 {sum(y==1)}）")
print(f"   每顆有 6 個特徵：R/G/B 的平均值與標準差")
print(f"   訓練用 {len(X_train)} 顆、測試用 {len(X_test)} 顆")

## 4-1：線性 SVM——電腦找出來的那條線

對每個顏色（R/G/B），讓 SVM 在「平均值 vs 標準差」的平面上找出分隔線。

圖上會出現：
- **實線**：SVM 找到的分隔線（決策邊界）
- **兩條虛線**：這條「最寬的路」的兩邊（margin）
- **被圈起來的點**：剛好踩在路邊、決定這條線怎麼畫的豆子，叫**支持向量**
  （support vector，SVM 的名字就是從這來的）

看看三個顏色裡，哪一個分得最乾淨？

In [ ]:
# ===== 4-1 default：線性 SVM，三通道各一條線，畫回散布圖 =====
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === 可以改 ===
C_VALUE    = 1.0    # 等一下會用滑桿玩它，先用 1
N_SHOW     = 60     # 每類畫幾顆點（訓練還是用全部，這只是畫圖抽樣）
ROW_HEIGHT = 480    # 每一張圖的高度，想更高就改大
# ==============

rng_show = np.random.default_rng(7)
show_idx = np.concatenate([
    rng_show.choice(np.where(y == 0)[0], N_SHOW, replace=False),
    rng_show.choice(np.where(y == 1)[0], N_SHOW, replace=False)])
show_set = set(show_idx.tolist())

fig = make_subplots(rows=3, cols=1,
                    subplot_titles=[channel_names[c] for c in range(3)],
                    vertical_spacing=0.12)

for c in range(3):
    Xc = X[:, [c, c + 3]]                       # (平均_c, 標準差_c)
    clf = SVC(kernel="linear", C=C_VALUE).fit(Xc, y)
    w, b = clf.coef_[0], clf.intercept_[0]

    # 抽樣的點
    for lbl, color, name in [(0, "#4169e1", "好豆"), (1, "#dc143c", "壞豆")]:
        idx = [i for i in show_idx if y[i] == lbl]
        fig.add_trace(go.Scatter(
            x=Xc[idx, 0], y=Xc[idx, 1], mode="markers",
            marker=dict(size=6, color=color, opacity=0.6),
            name=name, legendgroup=name, showlegend=(c == 0)),
            row=c + 1, col=1)

    # 支持向量圈起來（只圈有畫出來的那些）
    sv_show = [i for i in clf.support_ if i in show_set]
    fig.add_trace(go.Scatter(
        x=Xc[sv_show, 0], y=Xc[sv_show, 1], mode="markers",
        marker=dict(size=11, color="rgba(0,0,0,0)",
                    line=dict(color="#333", width=1.5)),
        name="支持向量", legendgroup="sv", showlegend=(c == 0)),
        row=c + 1, col=1)

    # 決策線 + 兩條 margin（在資料範圍內畫）
    xs = np.array([Xc[:, 0].min(), Xc[:, 0].max()])
    def line_y(off): return -(w[0] * xs + b - off) / w[1]
    fig.add_trace(go.Scatter(x=xs, y=line_y(0), mode="lines",
        line=dict(color="black", width=2), name="分隔線",
        legendgroup="line", showlegend=(c == 0)), row=c + 1, col=1)
    for off in (1, -1):
        fig.add_trace(go.Scatter(x=xs, y=line_y(off), mode="lines",
            line=dict(color="gray", width=1, dash="dash"),
            name="margin", legendgroup="m",
            showlegend=(c == 0 and off == 1)), row=c + 1, col=1)

fig.update_layout(height=ROW_HEIGHT * 3,
                  title_text=f"線性 SVM（C={C_VALUE}）",
                  legend=dict(orientation="h", y=1.03))
for c in range(3):
    fig.update_xaxes(title_text="平均值", row=c + 1, col=1)
    fig.update_yaxes(title_text="標準差", row=c + 1, col=1)
fig.show()

### （備用）如果上面的圖跑不出來，執行這一格

效果一樣：三個顏色各一條 SVM 分隔線，點和前面的散布圖對得起來。

In [ ]:
# ===== 4-1 fallback：純前端 HTML 畫線性 SVM（跑不出 Plotly 時用這個）=====
from IPython.display import HTML, display

# 在 Python 算好三通道的：抽樣點、決策線、margin、支持向量
N_SHOW = 60
rng_show = np.random.default_rng(7)
show_idx = np.concatenate([
    rng_show.choice(np.where(y == 0)[0], N_SHOW, replace=False),
    rng_show.choice(np.where(y == 1)[0], N_SHOW, replace=False)])
show_set = set(show_idx.tolist())

panels = []
for c in range(3):
    Xc = X[:, [c, c + 3]]
    clf = SVC(kernel="linear", C=C_VALUE).fit(Xc, y)
    w, b = clf.coef_[0], clf.intercept_[0]
    sv_set = set(clf.support_.tolist())
    xs = [float(Xc[:, 0].min()), float(Xc[:, 0].max())]
    def line_y(off): return [float(-(w[0]*x + b - off)/w[1]) for x in xs]
    pts = [{"x": float(Xc[i,0]), "y": float(Xc[i,1]),
            "lbl": int(y[i]), "sv": (i in sv_set)} for i in show_idx]
    panels.append({
        "name": channel_names[c], "pts": pts, "xs": xs,
        "line": line_y(0), "mUp": line_y(1), "mDn": line_y(-1),
        "xrange": [float(Xc[:,0].min()), float(Xc[:,0].max())],
        "yrange": [float(Xc[:,1].min()), float(Xc[:,1].max())],
    })

payload = json.dumps(panels)

html = f"""
<div id="svmapp" style="font-family:sans-serif;"></div>
<script>
(function() {{
  const panels = {payload};
  const palette = {{0:"#4169e1", 1:"#dc143c"}};  // 0好豆 1壞豆
  const W=720, H=420, PADL=60, PADB=50, PADT=40, PADR=30;
  const SVGNS="http://www.w3.org/2000/svg";
  const app=document.getElementById("svmapp");

  panels.forEach(p => {{
    const xpad=(p.xrange[1]-p.xrange[0])*0.08+1;
    const ypad=(p.yrange[1]-p.yrange[0])*0.12+1;
    const xlo=p.xrange[0]-xpad, xhi=p.xrange[1]+xpad;
    const ylo=p.yrange[0]-ypad, yhi=p.yrange[1]+ypad;
    const sx=x=>PADL+(x-xlo)/(xhi-xlo)*(W-PADL-PADR);
    const sy=v=>H-PADB-(v-ylo)/(yhi-ylo)*(H-PADT-PADB);

    const svg=document.createElementNS(SVGNS,"svg");
    svg.setAttribute("width",W);svg.setAttribute("height",H);
    svg.style.cssText="border:1px solid #ddd;margin-bottom:18px;display:block;";
    app.appendChild(svg);

    function L(x1,y1,x2,y2,color,wid,dash){{
      const l=document.createElementNS(SVGNS,"line");
      l.setAttribute("x1",x1);l.setAttribute("y1",y1);
      l.setAttribute("x2",x2);l.setAttribute("y2",y2);
      l.setAttribute("stroke",color);l.setAttribute("stroke-width",wid);
      if(dash)l.setAttribute("stroke-dasharray",dash);
      svg.appendChild(l);}}
    function T(x,y,s,size,anchor,rot){{
      const t=document.createElementNS(SVGNS,"text");
      t.setAttribute("x",x);t.setAttribute("y",y);t.setAttribute("font-size",size||12);
      t.setAttribute("fill","#888");t.setAttribute("text-anchor",anchor||"middle");
      if(rot)t.setAttribute("transform",`rotate(-90 ${{x}} ${{y}})`);
      t.textContent=s;svg.appendChild(t);}}

    // 軸 + 標題
    L(PADL,PADT,PADL,H-PADB,"#999",1); L(PADL,H-PADB,W-PADR,H-PADB,"#999",1);
    T(W/2,24,p.name,15);
    T(W/2,H-10,"平均值（多亮）"); T(16,H/2,"標準差（多不均勻）",12,"middle",true);

    // margin 兩條虛線
    L(sx(p.xs[0]),sy(p.mUp[0]),sx(p.xs[1]),sy(p.mUp[1]),"gray",1,"5,4");
    L(sx(p.xs[0]),sy(p.mDn[0]),sx(p.xs[1]),sy(p.mDn[1]),"gray",1,"5,4");
    // 決策線
    L(sx(p.xs[0]),sy(p.line[0]),sx(p.xs[1]),sy(p.line[1]),"black",2.5);

    // 點（支持向量多畫一圈）
    p.pts.forEach(pt => {{
      if(pt.sv){{
        const ring=document.createElementNS(SVGNS,"circle");
        ring.setAttribute("cx",sx(pt.x));ring.setAttribute("cy",sy(pt.y));
        ring.setAttribute("r",8);ring.setAttribute("fill","none");
        ring.setAttribute("stroke","#333");ring.setAttribute("stroke-width",1.5);
        svg.appendChild(ring);
      }}
      const c=document.createElementNS(SVGNS,"circle");
      c.setAttribute("cx",sx(pt.x));c.setAttribute("cy",sy(pt.y));
      c.setAttribute("r",4);c.setAttribute("fill",palette[pt.lbl]);
      c.setAttribute("opacity",0.65);
      svg.appendChild(c);
    }});
  }});
}})();
</script>
"""
display(HTML(html))

## 4-2：三個顏色一起看，做出最終判斷

剛剛我們三個顏色各畫了一條線。但每一條線都只看「一種顏色」，
就像只用一隻眼睛看世界——一定會看走眼。

**先看問題在哪。** 下面讓 R、G、B 三個顏色**各自**分豆子，
把分錯的豆子用 ❌ 標出來。你會發現：每個顏色都有分錯的，
而且各自錯的地方還不一樣。

In [ ]:
# ===== 4-2 第1步：R/G/B 各自分豆子，各自標出分錯的 =====
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N_SHOW     = 60
ROW_HEIGHT = 480
rng_show = np.random.default_rng(7)
show_idx = np.concatenate([
    rng_show.choice(np.where(y == 0)[0], N_SHOW, replace=False),
    rng_show.choice(np.where(y == 1)[0], N_SHOW, replace=False)])

# 先算三通道各自的準確率，放進子圖標題
titles = []
clfs, preds = [], []
for c in range(3):
    clf = SVC(kernel="linear", C=1.0).fit(X[:, [c, c+3]], y)
    pred = clf.predict(X[:, [c, c+3]])
    clfs.append(clf); preds.append(pred)
    titles.append(f"只看 {channel_names[c]}：準確率 {(pred==y).mean()*100:.1f}%")

fig = make_subplots(rows=3, cols=1, subplot_titles=titles, vertical_spacing=0.1)

for c in range(3):
    Xc = X[:, [c, c+3]]
    clf, pred = clfs[c], preds[c]

    # 正確分類的點
    for lbl, color, name in [(0, "#4169e1", "好豆"), (1, "#dc143c", "壞豆")]:
        idx = [i for i in show_idx if y[i] == lbl and pred[i] == y[i]]
        fig.add_trace(go.Scatter(x=Xc[idx,0], y=Xc[idx,1], mode="markers",
            marker=dict(size=7, color=color, opacity=0.6),
            name=name, legendgroup=name, showlegend=(c==0)), row=c+1, col=1)

    # 分錯的點：大叉叉
    wrong = [i for i in show_idx if pred[i] != y[i]]
    fig.add_trace(go.Scatter(x=Xc[wrong,0], y=Xc[wrong,1], mode="markers",
        marker=dict(size=14, color="black", symbol="x", line=dict(width=2)),
        name="分錯了 ❌", legendgroup="wrong", showlegend=(c==0)), row=c+1, col=1)

    # 決策線
    w, b = clf.coef_[0], clf.intercept_[0]
    xs = np.array([Xc[:,0].min(), Xc[:,0].max()])
    fig.add_trace(go.Scatter(x=xs, y=-(w[0]*xs+b)/w[1], mode="lines",
        line=dict(color="black", width=2), name="分隔線",
        legendgroup="line", showlegend=(c==0)), row=c+1, col=1)

fig.update_layout(height=ROW_HEIGHT*3, title_text="三個顏色各自分豆子，都會分錯一些",
                  legend=dict(orientation="h", y=1.03))
for c in range(3):
    fig.update_xaxes(title_text="平均值", row=c+1, col=1)
    fig.update_yaxes(title_text="標準差", row=c+1, col=1)
fig.show()

### （備用）上面跑不出來的話，執行這一格

In [ ]:
# ===== 4-2 第1步 fallback：純前端 HTML =====
# ===== 4-2 第1步 fallback：純前端 HTML，R/G/B 三張 =====
from IPython.display import HTML, display

N_SHOW = 60
rng_show = np.random.default_rng(7)
show_idx = np.concatenate([
    rng_show.choice(np.where(y == 0)[0], N_SHOW, replace=False),
    rng_show.choice(np.where(y == 1)[0], N_SHOW, replace=False)])

panels = []
for c in range(3):
    clf = SVC(kernel="linear", C=1.0).fit(X[:, [c, c+3]], y)
    pred = clf.predict(X[:, [c, c+3]])
    Xc = X[:, [c, c+3]]
    w, b = clf.coef_[0], clf.intercept_[0]
    xs = [float(Xc[:,0].min()), float(Xc[:,0].max())]
    panels.append({
        "name": channel_names[c],
        "acc": float((pred == y).mean()),
        "pts": [{"x": float(Xc[i,0]), "y": float(Xc[i,1]),
                 "lbl": int(y[i]), "wrong": bool(pred[i] != y[i])}
                for i in show_idx],
        "xs": xs,
        "line": [float(-(w[0]*x + b)/w[1]) for x in xs],
        "xrange": [float(Xc[:,0].min()), float(Xc[:,0].max())],
        "yrange": [float(Xc[:,1].min()), float(Xc[:,1].max())],
    })

payload = json.dumps(panels)

html = f"""
<div id="single3" style="font-family:sans-serif;"></div>
<script>
(function() {{
  const panels = {payload};
  const palette = {{0:"#4169e1", 1:"#dc143c"}};
  const W=720, H=460, PADL=60, PADB=50, PADT=46, PADR=30;
  const SVGNS="http://www.w3.org/2000/svg";
  const app=document.getElementById("single3");

  panels.forEach(D => {{
    const xpad=(D.xrange[1]-D.xrange[0])*0.08+1, ypad=(D.yrange[1]-D.yrange[0])*0.12+1;
    const xlo=D.xrange[0]-xpad, xhi=D.xrange[1]+xpad;
    const ylo=D.yrange[0]-ypad, yhi=D.yrange[1]+ypad;
    const sx=x=>PADL+(x-xlo)/(xhi-xlo)*(W-PADL-PADR);
    const sy=v=>H-PADB-(v-ylo)/(yhi-ylo)*(H-PADT-PADB);

    const svg=document.createElementNS(SVGNS,"svg");
    svg.setAttribute("width",W);svg.setAttribute("height",H);
    svg.style.cssText="border:1px solid #ddd;display:block;margin-bottom:18px;";
    app.appendChild(svg);

    function L(x1,y1,x2,y2,col,wid){{const l=document.createElementNS(SVGNS,"line");
      l.setAttribute("x1",x1);l.setAttribute("y1",y1);l.setAttribute("x2",x2);l.setAttribute("y2",y2);
      l.setAttribute("stroke",col);l.setAttribute("stroke-width",wid);svg.appendChild(l);}}
    function T(x,y,s,size,anchor,rot){{const t=document.createElementNS(SVGNS,"text");
      t.setAttribute("x",x);t.setAttribute("y",y);t.setAttribute("font-size",size||12);
      t.setAttribute("fill","#888");t.setAttribute("text-anchor",anchor||"middle");
      if(rot)t.setAttribute("transform",`rotate(-90 ${{x}} ${{y}})`);
      t.textContent=s;svg.appendChild(t);}}

    L(PADL,PADT,PADL,H-PADB,"#999",1); L(PADL,H-PADB,W-PADR,H-PADB,"#999",1);
    T(W/2,26,`只看「${{D.name}}」一個顏色：準確率 ${{(D.acc*100).toFixed(1)}}%`,15);
    T(W/2,H-10,"平均值（多亮）"); T(16,H/2,"標準差（多不均勻）",12,"middle",true);

    L(sx(D.xs[0]),sy(D.line[0]),sx(D.xs[1]),sy(D.line[1]),"black",2.5);

    D.pts.forEach(pt => {{
      if(pt.wrong){{
        const x=sx(pt.x), v=sy(pt.y), r=7;
        L(x-r,v-r,x+r,v+r,"black",2.5); L(x-r,v+r,x+r,v-r,"black",2.5);
      }} else {{
        const c=document.createElementNS(SVGNS,"circle");
        c.setAttribute("cx",sx(pt.x));c.setAttribute("cy",sy(pt.y));
        c.setAttribute("r",5);c.setAttribute("fill",palette[pt.lbl]);
        c.setAttribute("opacity",0.65);svg.appendChild(c);
      }}
    }});
  }});
}})();
</script>
"""
display(HTML(html))

## 4-2 第二步：三個顏色「一起看」

剛剛每個顏色單獨看，都會分錯一些。現在讓 SVM **同時看 R、G、B 全部資訊**
（每顆豆子有 6 個數字：三個顏色的平均值和標準差），一次找出最好的判斷。

這次我們很誠實：用**沒看過的豆子**（測試組）來打分數，
看「一起看」到底有沒有比「只看一個」厲害。

In [ ]:
# ===== 4-2 第2步：三個一起看 vs 只看一個，準確率對比 =====
import plotly.graph_objects as go

# 三個單通道（在測試集上評分）
acc = []
for c in range(3):
    clf = SVC(kernel="linear", C=1.0).fit(X_train[:, [c, c+3]], y_train)
    acc.append(clf.score(X_test[:, [c, c+3]], y_test))

# 六維：三個顏色一起看
full_clf = SVC(kernel="linear", C=1.0).fit(X_train, y_train)
acc_full = full_clf.score(X_test, y_test)

labels = [f"只看{n}" for n in ["紅", "綠", "藍"]] + ["三個一起看"]
values = acc + [acc_full]
colors = ["#dc143c", "#2e8b57", "#4169e1", "#ff8c00"]

fig = go.Figure(go.Bar(
    x=labels, y=[v*100 for v in values],
    marker_color=colors,
    text=[f"{v*100:.1f}%" for v in values], textposition="outside"))
fig.update_layout(height=460,
    title_text="在「沒看過的豆子」上的準確率",
    yaxis_title="準確率 (%)", yaxis_range=[80, 100])
fig.show()

print(f"只看一個顏色，最高 {max(acc)*100:.1f}%")
print(f"三個顏色一起看：{acc_full*100:.1f}%  ← 厲害多了！")

## 4-3：打開這個模型，看它在想什麼

「三個一起看」準確率 97%，但它到底**最看重哪個特徵**？

線性 SVM 給每個特徵一個**權重**，權重的絕對值越大，
代表這個特徵對判斷的「影響力」越大。把六個權重畫出來看看：

In [ ]:
# ===== 4-3 第1步：看模型的六個權重（依顏色通道上色）=====
import plotly.graph_objects as go

full_clf = SVC(kernel="linear", C=1.0).fit(X_train, y_train)
w = full_clf.coef_[0]

feat_names = ["R 平均", "G 平均", "B 平均", "R 標準差", "G 標準差", "B 標準差"]
# 依通道上色：R紅 G綠 B藍
bar_colors = ["#dc143c", "#2e8b57", "#4169e1", "#dc143c", "#2e8b57", "#4169e1"]

fig = go.Figure(go.Bar(
    x=feat_names, y=w, marker_color=bar_colors,
    text=[f"{v:+.2f}" for v in w], textposition="outside"))
fig.update_layout(height=460,
    title_text="模型的六個權重（紅/綠/藍代表三個顏色通道）",
    yaxis_title="權重", xaxis_title="特徵")
fig.add_hline(y=0, line_color="#888")
fig.update_layout(height=600,
    title_text="模型的六個權重（紅/綠/藍代表三個顏色通道）",
    yaxis_title="權重", xaxis_title="特徵")
fig.show()

top = feat_names[np.argmax(np.abs(w))]
print(f"權重絕對值最大的是：{top}")
print(f"所以……它最重要？我們乾脆只用「{top}」設一條線來分豆子就好？")

### （備用）上面跑不出來的話，執行這一格

In [ ]:
# ===== 4-3 第1步 fallback：純前端 HTML 長條圖 =====
from IPython.display import HTML, display

full_clf = SVC(kernel="linear", C=1.0).fit(X_train, y_train)
w = full_clf.coef_[0]

data = {
    "names": ["R 平均", "G 平均", "B 平均", "R 標準差", "G 標準差", "B 標準差"],
    "w": [float(v) for v in w],
    "colors": ["#dc143c", "#2e8b57", "#4169e1", "#dc143c", "#2e8b57", "#4169e1"],
}
payload = json.dumps(data)

html = f"""
<div id="wbar" style="font-family:sans-serif;"></div>
<script>
(function() {{
  const D = {payload};
  const W=900, H=600, PADL=70, PADB=60, PADT=30, PADR=30;
  const SVGNS="http://www.w3.org/2000/svg";
  const app=document.getElementById("wbar");
  const svg=document.createElementNS(SVGNS,"svg");
  svg.setAttribute("width",W);svg.setAttribute("height",H);
  svg.style.cssText="border:1px solid #ddd;display:block;";
  app.appendChild(svg);

  const wmax=Math.max(...D.w.map(Math.abs))*1.25;
  const ylo=-wmax, yhi=wmax;
  const sy=v=>PADT+(yhi-v)/(yhi-ylo)*(H-PADT-PADB);
  const n=D.w.length;
  const plotW=W-PADL-PADR, slot=plotW/n, bw=slot*0.6;

  function L(x1,y1,x2,y2,col,wid){{const l=document.createElementNS(SVGNS,"line");
    l.setAttribute("x1",x1);l.setAttribute("y1",y1);l.setAttribute("x2",x2);l.setAttribute("y2",y2);
    l.setAttribute("stroke",col);l.setAttribute("stroke-width",wid);svg.appendChild(l);}}
  function T(x,y,s,size,col,anchor){{const t=document.createElementNS(SVGNS,"text");
    t.setAttribute("x",x);t.setAttribute("y",y);t.setAttribute("font-size",size||13);
    t.setAttribute("fill",col||"#333");t.setAttribute("text-anchor",anchor||"middle");
    t.textContent=s;svg.appendChild(t);}}

  // y 軸刻度
  T(36,PADT+5,"權重",13,"#888");
  [-1,-0.5,0,0.5,1].forEach(v=>{{
    if(v>=ylo&&v<=yhi){{
      L(PADL,sy(v),W-PADR,sy(v),"#eee",1);
      T(PADL-10,sy(v)+4,v.toFixed(1),12,"#888","end");
    }}
  }});
  L(PADL,sy(0),W-PADR,sy(0),"#888",1.5);  // 零線

  // bar
  D.w.forEach((v,i)=>{{
    const cx=PADL+slot*(i+0.5);
    const y0=sy(0), y1=sy(v);
    const r=document.createElementNS(SVGNS,"rect");
    r.setAttribute("x",cx-bw/2);r.setAttribute("y",Math.min(y0,y1));
    r.setAttribute("width",bw);r.setAttribute("height",Math.abs(y1-y0));
    r.setAttribute("fill",D.colors[i]);r.setAttribute("opacity",0.85);
    svg.appendChild(r);
    // 數值
    const sign=v>=0?"+":"";
    T(cx, v>=0? y1-8 : y1+18, sign+v.toFixed(2), 13, "#333");
    // x 標籤
    T(cx, H-PADB+22, D.names[i], 13, "#555");
  }});
}})();
</script>
"""
display(HTML(html))

top = data["names"][int(np.argmax(np.abs(w)))]
print(f"權重絕對值最大的是：{top}")
print(f"所以……它最重要？我們乾脆只用「{top}」設一條線來分豆子就好？")

## 4-3 第二步：等等，權重最大 ≠ 最重要！

如果「B 標準差」真的最重要，那只用它一個應該就能分得不錯吧？
我們來實測：把六個特徵**各自單獨**拿來分豆子，看各能到幾分。

In [ ]:
# ===== 4-3 第2步：六個特徵各自單用，能分多準？=====
import plotly.graph_objects as go

feat_names = ["R 平均", "G 平均", "B 平均", "R 標準差", "G 標準差", "B 標準差"]
bar_colors = ["#dc143c", "#2e8b57", "#4169e1", "#dc143c", "#2e8b57", "#4169e1"]

single_acc = []
for i in range(6):
    clf = SVC(kernel="linear", C=1.0).fit(X_train[:, [i]], y_train)
    single_acc.append(clf.score(X_test[:, [i]], y_test) * 100)

full_acc = SVC(kernel="linear", C=1.0).fit(X_train, y_train).score(X_test, y_test) * 100

# 加一根「全部一起」當對照
x_labels = feat_names + ["全部一起"]
y_vals   = single_acc + [full_acc]
colors   = bar_colors + ["#ff8c00"]

fig = go.Figure(go.Bar(
    x=x_labels, y=y_vals, marker_color=colors,
    text=[f"{v:.1f}%" for v in y_vals], textposition="outside"))
fig.add_hline(y=50, line_dash="dash", line_color="#888",
              annotation_text="亂猜 50%", annotation_position="right")
fig.update_layout(height=560,
    title_text="各特徵單獨使用的準確率（權重最大的 B 標準差，竟然最低！）",
    yaxis_title="準確率 (%)", yaxis_range=[40, 100])
fig.show()

print(f"權重最大的「B 標準差」單用只有 {single_acc[5]:.1f}% —— 跟丟銅板差不多！")
print(f"反而權重很小的「B 平均」單用有 {single_acc[2]:.1f}%。")
print("結論：權重的大小，不能直接拿來比哪個特徵重要。為什麼？下一格揭曉。")

### （備用）上面跑不出來的話，執行這一格

In [ ]:
# ===== 4-3 第2步 fallback：純前端 HTML 長條圖 =====
from IPython.display import HTML, display

single_acc = [SVC(kernel="linear", C=1.0).fit(X_train[:, [i]], y_train)
              .score(X_test[:, [i]], y_test) * 100 for i in range(6)]
full_acc = SVC(kernel="linear", C=1.0).fit(X_train, y_train).score(X_test, y_test) * 100

data = {
    "labels": ["R 平均", "G 平均", "B 平均", "R 標準差", "G 標準差", "B 標準差", "全部一起"],
    "vals": [float(a) for a in single_acc] + [float(full_acc)],
    "colors": ["#dc143c", "#2e8b57", "#4169e1", "#dc143c", "#2e8b57", "#4169e1", "#ff8c00"],
}
payload = json.dumps(data)

html = f"""
<div id="accbar" style="font-family:sans-serif;"></div>
<script>
(function() {{
  const D = {payload};
  const W=1000, H=560, PADL=60, PADB=60, PADT=30, PADR=70;
  const SVGNS="http://www.w3.org/2000/svg";
  const app=document.getElementById("accbar");
  const svg=document.createElementNS(SVGNS,"svg");
  svg.setAttribute("width",W);svg.setAttribute("height",H);
  svg.style.cssText="border:1px solid #ddd;display:block;";
  app.appendChild(svg);

  const ylo=40, yhi=100;
  const sy=v=>PADT+(yhi-v)/(yhi-ylo)*(H-PADT-PADB);
  const n=D.vals.length, plotW=W-PADL-PADR, slot=plotW/n, bw=slot*0.6;

  function L(x1,y1,x2,y2,col,wid,dash){{const l=document.createElementNS(SVGNS,"line");
    l.setAttribute("x1",x1);l.setAttribute("y1",y1);l.setAttribute("x2",x2);l.setAttribute("y2",y2);
    l.setAttribute("stroke",col);l.setAttribute("stroke-width",wid);
    if(dash)l.setAttribute("stroke-dasharray",dash);svg.appendChild(l);}}
  function T(x,y,s,size,col,anchor){{const t=document.createElementNS(SVGNS,"text");
    t.setAttribute("x",x);t.setAttribute("y",y);t.setAttribute("font-size",size||13);
    t.setAttribute("fill",col||"#333");t.setAttribute("text-anchor",anchor||"middle");
    t.textContent=s;svg.appendChild(t);}}

  T(30,PADT+5,"準確率(%)",12,"#888");
  [40,50,60,70,80,90,100].forEach(v=>{{
    L(PADL,sy(v),W-PADR,sy(v),"#eee",1);
    T(PADL-10,sy(v)+4,v,12,"#888","end");
  }});
  // 亂猜 50% 線
  L(PADL,sy(50),W-PADR,sy(50),"#888",1.5,"6,4");
  T(W-PADR+8,sy(50)+4,"亂猜 50%",12,"#888","start");

  D.vals.forEach((v,i)=>{{
    const cx=PADL+slot*(i+0.5);
    const r=document.createElementNS(SVGNS,"rect");
    r.setAttribute("x",cx-bw/2);r.setAttribute("y",sy(v));
    r.setAttribute("width",bw);r.setAttribute("height",sy(ylo)-sy(v));
    r.setAttribute("fill",D.colors[i]);r.setAttribute("opacity",0.88);
    svg.appendChild(r);
    T(cx, sy(v)-8, v.toFixed(1)+"%", 13, "#333");
    T(cx, H-PADB+22, D.labels[i], 12, "#555");
  }});
}})();
</script>
"""
display(HTML(html))

print(f"權重最大的「B 標準差」單用只有 {single_acc[5]:.1f}% —— 跟丟銅板差不多！")
print(f"反而權重很小的「B 平均」單用有 {single_acc[2]:.1f}%。")
print("結論：權重的大小，不能直接拿來比哪個特徵重要。為什麼？下一格揭曉。")

## 4-3 第三步：揭曉——權重大，不代表單獨好用

把每個特徵的「權重」和「單獨使用的準確率」擺在一起比（都換算成 0~1 方便比高低）：

In [ ]:
# ===== 4-3 第3步：權重 vs 單用準確率，兩件不同的事 =====
import plotly.graph_objects as go

feat_names = ["R 平均", "G 平均", "B 平均", "R 標準差", "G 標準差", "B 標準差"]

w_abs = np.abs(SVC(kernel="linear", C=1.0).fit(X_train, y_train).coef_[0])
single_acc = np.array([
    SVC(kernel="linear", C=1.0).fit(X_train[:, [i]], y_train)
    .score(X_test[:, [i]], y_test) for i in range(6)])

# 各自歸一化到 0~1，只比相對高低
w_norm   = w_abs / w_abs.max()
acc_norm = (single_acc - 0.5) / (single_acc.max() - 0.5)  # 以亂猜50%為基準

fig = go.Figure()
fig.add_trace(go.Bar(x=feat_names, y=w_norm, name="權重（影響力）",
                     marker_color="#ff8c00"))
fig.add_trace(go.Bar(x=feat_names, y=acc_norm, name="單獨使用的準確率",
                     marker_color="#888"))
fig.update_layout(height=520, barmode="group",
    title_text="權重 ≠ 單獨好用（都換算成 0~1）",
    yaxis_title="相對高低", legend=dict(orientation="h", y=1.08))
fig.show()

print("看「B 標準差」：權重最高（橘色最長），但單獨用時最爛（灰色最短）。")
print("看「B 平均」：權重幾乎是 0，但單獨用時反而最好。")
print()
print("為什麼？因為一個特徵的價值，要看它跟『其他特徵搭配』時的貢獻——")
print("B 標準差自己沒區辨力，但它能在團隊裡幫忙修正別人的錯誤，所以模型給它大權重。")
print("這就是為什麼「全部一起看」最強：六個特徵互相補位，像一支球隊。")

### （備用）上面跑不出來的話，執行這一格

In [ ]:
# ===== 4-3 第3步 fallback：純前端 HTML 分組長條圖 =====
from IPython.display import HTML, display

w_abs = np.abs(SVC(kernel="linear", C=1.0).fit(X_train, y_train).coef_[0])
single_acc = np.array([
    SVC(kernel="linear", C=1.0).fit(X_train[:, [i]], y_train)
    .score(X_test[:, [i]], y_test) for i in range(6)])
w_norm   = (w_abs / w_abs.max()).tolist()
acc_norm = ((single_acc - 0.5) / (single_acc.max() - 0.5)).tolist()

data = {
    "names": ["R 平均", "G 平均", "B 平均", "R 標準差", "G 標準差", "B 標準差"],
    "w": [float(v) for v in w_norm],
    "acc": [float(v) for v in acc_norm],
}
payload = json.dumps(data)

html = f"""
<div id="cmpbar" style="font-family:sans-serif;"></div>
<script>
(function() {{
  const D = {payload};
  const W=920, H=520, PADL=60, PADB=80, PADT=30, PADR=30;
  const SVGNS="http://www.w3.org/2000/svg";
  const app=document.getElementById("cmpbar");
  const svg=document.createElementNS(SVGNS,"svg");
  svg.setAttribute("width",W);svg.setAttribute("height",H);
  svg.style.cssText="border:1px solid #ddd;display:block;";
  app.appendChild(svg);

  const ylo=0, yhi=1.05;
  const sy=v=>PADT+(yhi-v)/(yhi-ylo)*(H-PADT-PADB);
  const n=D.names.length, plotW=W-PADL-PADR, slot=plotW/n;
  const bw=slot*0.32;

  function L(x1,y1,x2,y2,col,wid){{const l=document.createElementNS(SVGNS,"line");
    l.setAttribute("x1",x1);l.setAttribute("y1",y1);l.setAttribute("x2",x2);l.setAttribute("y2",y2);
    l.setAttribute("stroke",col);l.setAttribute("stroke-width",wid);svg.appendChild(l);}}
  function T(x,y,s,size,col,anchor){{const t=document.createElementNS(SVGNS,"text");
    t.setAttribute("x",x);t.setAttribute("y",y);t.setAttribute("font-size",size||13);
    t.setAttribute("fill",col||"#333");t.setAttribute("text-anchor",anchor||"middle");
    t.textContent=s;svg.appendChild(t);}}
  function bar(x,v,col){{const r=document.createElementNS(SVGNS,"rect");
    r.setAttribute("x",x);r.setAttribute("y",sy(v));r.setAttribute("width",bw);
    r.setAttribute("height",sy(0)-sy(v));r.setAttribute("fill",col);
    r.setAttribute("opacity",0.9);svg.appendChild(r);}}

  T(30,PADT+5,"相對高低",12,"#888");
  [0,0.25,0.5,0.75,1].forEach(v=>{{
    L(PADL,sy(v),W-PADR,sy(v),"#eee",1); T(PADL-10,sy(v)+4,v,12,"#888","end");
  }});
  L(PADL,sy(0),W-PADR,sy(0),"#888",1.5);

  D.names.forEach((nm,i)=>{{
    const cx=PADL+slot*(i+0.5);
    bar(cx-bw-2, D.w[i],   "#ff8c00");  // 權重
    bar(cx+2,    D.acc[i], "#888");     // 單用準確率
    T(cx, H-PADB+22, nm, 12, "#555");
  }});

  // 圖例
  function legend(x,col,txt){{const r=document.createElementNS(SVGNS,"rect");
    r.setAttribute("x",x);r.setAttribute("y",H-26);r.setAttribute("width",14);
    r.setAttribute("height",14);r.setAttribute("fill",col);svg.appendChild(r);
    T(x+20,H-14,txt,13,"#555","start");}}
  legend(PADL+60,"#ff8c00","權重（影響力）");
  legend(PADL+240,"#888","單獨使用的準確率");
}})();
</script>
"""
display(HTML(html))

print("看「B 標準差」：權重最高（橘色最長），但單獨用時最爛（灰色最短）。")
print("看「B 平均」：權重幾乎是 0，但單獨用時反而最好。")
print()
print("為什麼？因為一個特徵的價值，要看它跟『其他特徵搭配』時的貢獻——")
print("B 標準差自己沒區辨力，但它能在團隊裡幫忙修正別人的錯誤，所以模型給它大權重。")
print("這就是為什麼「全部一起看」最強：六個特徵互相補位，像一支球隊。")

# 步驟 5：來吧，用你的模型辨識真正的豆子！

前面我們把模型看透了。現在直接用它——
這就是一條完整的「機器學習辨識生產線」：

**抽一顆豆子 → 算出 6 個特徵 → 丟進 SVM 模型 → 得到判斷**

抽到的都是模型**沒看過**的豆子（測試組），所以這是它真正的實力。

In [ ]:
# ===== 步驟5：互動辨識生產線 =====
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output
import base64
from sklearn.model_selection import train_test_split

# 記住每顆豆子是誰（沿用 Cell 10 抽樣的 g_idx / b_idx）
img_dirs = {"good": GOOD_DIR, "bad": BAD_DIR}
def bean_image_b64(label, fname):
    ok, buf = cv2.imencode(".jpg", cv2.imread(str(img_dirs[label] / fname)))
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()

bean_label, bean_fname = [], []
for i in g_idx:
    bean_label.append("good"); bean_fname.append(good_files[i])
for i in b_idx:
    bean_label.append("bad");  bean_fname.append(bad_files[i])

# 最終模型 + 測試組索引（抽豆子只從沒看過的 test 抽）
model = SVC(kernel="linear", C=1.0).fit(X_train, y_train)
_idx = np.arange(len(X))
_, _, _, _, _, test_idx = train_test_split(X, y, _idx, test_size=0.3,
                                            random_state=1, stratify=y)

state = {"pick": None}

btn_draw    = widgets.Button(description="🎲 抽一顆豆子", button_style="info",
                             layout=widgets.Layout(width="160px", height="44px"))
btn_predict = widgets.Button(description="🤖 用機器學習模型辨識！", button_style="warning",
                             layout=widgets.Layout(width="240px", height="44px"),
                             disabled=True)
out = widgets.Output()

def render(stage):
    i = state["pick"]
    label, fname, feat = bean_label[i], bean_fname[i], X[i]
    img = bean_image_b64(label, fname)
    feat_str = "、".join(f"{v:.0f}" for v in feat)

    result_html = ""
    if stage == "done":
        pred, true = model.predict([feat])[0], y[i]
        correct = (pred == true)
        pred_zh = "好豆" if pred == 0 else "壞豆"
        true_zh = "好豆" if true == 0 else "壞豆"
        badge = "✅ 答對了！" if correct else "❌ 這次猜錯了"
        bg, bd = ("#e8f5e9", "#43a047") if correct else ("#ffebee", "#e53935")
        result_html = f"""
        <div style="font-size:34px;color:#999;">⬇</div>
        <div style="background:{bg};border:2px solid {bd};border-radius:12px;
                    padding:16px;max-width:300px;margin:0 auto;">
          <div style="font-size:22px;font-weight:bold;">模型判斷：{pred_zh}</div>
          <div style="font-size:14px;color:#666;margin-top:4px;">正確答案：{true_zh}</div>
          <div style="font-size:20px;margin-top:8px;">{badge}</div>
        </div>"""

    with out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="font-family:sans-serif;text-align:center;">
          <img src="{img}" width="130" style="border-radius:8px;border:1px solid #ccc;">
          <div style="font-size:13px;color:#666;margin:6px 0;">這顆豆子</div>
          <div style="font-size:34px;color:#999;">⬇</div>
          <div style="font-size:12px;color:#888;">算出 6 個特徵：{feat_str}</div>
          <div style="font-size:34px;color:#999;">⬇</div>
          <div style="display:inline-block;background:#fff8e1;border:2px solid #ff8c00;
                      border-radius:12px;padding:14px 24px;">
            <div style="font-size:30px;">🤖</div>
            <div style="font-weight:bold;">SVM 模型</div>
            <div style="font-size:11px;color:#888;">看 6 個特徵做判斷</div>
          </div>
          {result_html}
        </div>"""))

def on_draw(_):
    state["pick"] = int(np.random.default_rng().choice(test_idx))
    btn_predict.disabled = False
    render("drawn")

def on_predict(_):
    if state["pick"] is not None:
        render("done")

btn_draw.on_click(on_draw)
btn_predict.on_click(on_predict)

display(widgets.HBox([btn_draw, btn_predict]))
display(out)
print("先按「抽一顆豆子」，再按「用機器學習模型辨識！」")

### （備用）上面的按鈕沒反應的話，執行這一格

In [ ]:
# ===== 步驟5 fallback：純前端 HTML 互動生產線 =====
from IPython.display import HTML, display
from sklearn.model_selection import train_test_split

img_dirs = {"good": GOOD_DIR, "bad": BAD_DIR}
bean_label, bean_fname = [], []
for i in g_idx:
    bean_label.append("good"); bean_fname.append(good_files[i])
for i in b_idx:
    bean_label.append("bad");  bean_fname.append(bad_files[i])

model = SVC(kernel="linear", C=1.0).fit(X_train, y_train)
_idx = np.arange(len(X))
_, _, _, _, _, test_idx = train_test_split(X, y, _idx, test_size=0.3,
                                            random_state=1, stratify=y)

def small_b64(label, fname):
    img = cv2.imread(str(img_dirs[label] / fname))
    h, w = img.shape[:2]; s = 100 / max(h, w)
    img = cv2.resize(img, (int(w*s), int(h*s)))
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()

pool = []
for i in test_idx:
    pool.append({
        "img": small_b64(bean_label[i], bean_fname[i]),
        "feat": [round(float(v)) for v in X[i]],
        "pred": int(model.predict([X[i]])[0]),
        "true": int(y[i]),
    })
payload = json.dumps(pool)

html = f"""
<div style="font-family:sans-serif;">
  <div style="margin-bottom:12px;">
    <button id="bDraw" style="font-size:15px;padding:10px 18px;border:none;border-radius:8px;
            background:#2196f3;color:#fff;cursor:pointer;">🎲 抽一顆豆子</button>
    <button id="bPred" style="font-size:15px;padding:10px 18px;border:none;border-radius:8px;
            background:#ff8c00;color:#fff;cursor:pointer;opacity:0.5;" disabled>🤖 用機器學習模型辨識！</button>
  </div>
  <div id="stage" style="text-align:center;"></div>
</div>
<script>
(function() {{
  const pool = {payload};
  let pick = null;
  const stage = document.getElementById("stage");
  const bDraw = document.getElementById("bDraw");
  const bPred = document.getElementById("bPred");

  function arrow() {{ return '<div style="font-size:34px;color:#999;">⬇</div>'; }}

  function renderDrawn() {{
    const b = pool[pick];
    stage.innerHTML = `
      <img src="${{b.img}}" width="130" style="border-radius:8px;border:1px solid #ccc;">
      <div style="font-size:13px;color:#666;margin:6px 0;">這顆豆子</div>
      ${{arrow()}}
      <div style="font-size:12px;color:#888;">算出 6 個特徵：${{b.feat.join("、")}}</div>
      ${{arrow()}}
      <div style="display:inline-block;background:#fff8e1;border:2px solid #ff8c00;
                  border-radius:12px;padding:14px 24px;">
        <div style="font-size:30px;">🤖</div>
        <div style="font-weight:bold;">SVM 模型</div>
        <div style="font-size:11px;color:#888;">看 6 個特徵做判斷</div>
      </div>
      <div id="result"></div>`;
  }}

  function renderDone() {{
    const b = pool[pick];
    const correct = (b.pred === b.true);
    const predZh = b.pred === 0 ? "好豆" : "壞豆";
    const trueZh = b.true === 0 ? "好豆" : "壞豆";
    const badge = correct ? "✅ 答對了！" : "❌ 這次猜錯了";
    const bg = correct ? "#e8f5e9" : "#ffebee";
    const bd = correct ? "#43a047" : "#e53935";
    document.getElementById("result").innerHTML = `
      ${{arrow()}}
      <div style="background:${{bg}};border:2px solid ${{bd}};border-radius:12px;
                  padding:16px;max-width:300px;margin:0 auto;">
        <div style="font-size:22px;font-weight:bold;">模型判斷：${{predZh}}</div>
        <div style="font-size:14px;color:#666;margin-top:4px;">正確答案：${{trueZh}}</div>
        <div style="font-size:20px;margin-top:8px;">${{badge}}</div>
      </div>`;
  }}

  bDraw.onclick = () => {{
    pick = Math.floor(Math.random() * pool.length);
    bPred.disabled = false; bPred.style.opacity = "1";
    renderDrawn();
  }};
  bPred.onclick = () => {{ if (pick !== null) renderDone(); }};
}})();
</script>
"""
display(HTML(html))

# 步驟 6：見證奇蹟——一眨眼判完全部豆子

剛剛一顆一顆抽很慢。但對機器來說，一顆和一千顆沒差別。

按下按鈕，讓模型**一次判完全部 1000 顆**，看看它要花多久。

In [ ]:
# ===== 步驟6：秒速辨識全部豆子（滑桿即時調牆數）=====
from IPython.display import HTML, display
import time, base64

model = SVC(kernel="linear", C=1.0).fit(X_train, y_train)

img_dirs = {"good": GOOD_DIR, "bad": BAD_DIR}
bean_label, bean_fname = [], []
for i in g_idx:
    bean_label.append("good"); bean_fname.append(good_files[i])
for i in b_idx:
    bean_label.append("bad");  bean_fname.append(bad_files[i])

# 真正計時辨識全部
t0 = time.time()
pred = model.predict(X)
dt_ms = (time.time() - t0) * 1000
acc = (pred == y).mean() * 100
n_wrong = int((pred != y).sum())
human_min = len(X) * 2 / 60

def tiny_b64(label, fname):
    img = cv2.imread(str(img_dirs[label] / fname))
    h, w = img.shape[:2]; s = 48 / max(h, w)
    img = cv2.resize(img, (int(w*s), int(h*s)))
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 70])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()

# 全部 1000 顆做成資料（先排錯的，讓任何牆數都能看到一些 ❌）
wrong = np.where(pred != y)[0].tolist()
right = np.where(pred == y)[0].tolist()
order = wrong + right
cells = [{"img": tiny_b64(bean_label[i], bean_fname[i]),
          "ok": bool(pred[i] == y[i])} for i in order]
payload = json.dumps(cells)

html = f"""
<div style="font-family:sans-serif;">
  <div style="display:flex;gap:30px;text-align:center;margin-bottom:16px;flex-wrap:wrap;">
    <div><div style="font-size:40px;font-weight:bold;color:#2196f3;">{len(X)}</div>
         <div style="color:#666;">顆豆子</div></div>
    <div><div style="font-size:40px;font-weight:bold;color:#ff8c00;">{dt_ms:.1f}<span style="font-size:18px;"> 毫秒</span></div>
         <div style="color:#666;">總共花的時間</div></div>
    <div><div style="font-size:40px;font-weight:bold;color:#43a047;">{acc:.1f}%</div>
         <div style="color:#666;">準確率（錯 {n_wrong} 顆）</div></div>
  </div>
  <div style="background:#e3f2fd;border-radius:8px;padding:10px;margin-bottom:14px;color:#333;">
    💡 同樣的工作，人一顆一顆看（每顆 2 秒）要花約 <b>{human_min:.0f} 分鐘</b>；
    機器只花了 <b>{dt_ms:.1f} 毫秒</b>。
  </div>
  <div style="margin-bottom:8px;">
    牆要放幾顆：<input id="wallRange" type="range" min="20" max="{len(X)}" value="300" step="10"
                 style="vertical-align:middle;width:300px;">
    <span id="wallNum" style="font-weight:bold;">300</span> 顆
    <span style="color:#888;font-size:12px;">（✅=判對 ❌=判錯）</span>
  </div>
  <div id="wall" style="display:flex;flex-wrap:wrap;gap:6px;"></div>
</div>
<script>
(function() {{
  const cells = {payload};
  const wall = document.getElementById("wall");
  const range = document.getElementById("wallRange");
  const num = document.getElementById("wallNum");

  function render(n) {{
    num.textContent = n;
    let html = "";
    for (let k = 0; k < n; k++) {{
      const c = cells[k];
      const border = c.ok ? "#43a047" : "#e53935";
      const mark = c.ok ? "✅" : "❌";
      html += `<div style="position:relative;">
        <img src="${{c.img}}" width="44" height="44"
             style="object-fit:cover;border-radius:4px;outline:2px solid ${{border}};">
        <span style="position:absolute;bottom:-2px;right:-2px;font-size:11px;">${{mark}}</span>
      </div>`;
    }}
    wall.innerHTML = html;
  }}

  range.oninput = () => render(parseInt(range.value));
  render(300);
}})();
</script>
"""
display(HTML(html))